# Exploratory Data Analysis - Scam Detection Dataset

This notebook explores the scam detection dataset and provides insights into the data distribution, patterns, and characteristics.

## Objectives
1. Load and inspect the dataset
2. Analyze data distribution
3. Explore text characteristics
4. Identify common patterns in scams
5. Visualize Australian-specific patterns

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.append('../..')

from src.preprocessing.text_cleaner import TextCleaner
from src.preprocessing.feature_extractor import FeatureExtractor
from src.preprocessing.australian_patterns import AustralianPatternDetector

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries loaded successfully")

## 1. Load Dataset

In [ ]:
# Load data
data_path = Path('../../data/raw/sample_scams.csv')

if not data_path.exists():
    print("⚠️ Dataset not found. Generating sample data...")
    !python ../../scripts/generate_sample_data.py

df = pd.read_csv(data_path)
print(f"📊 Loaded {len(df)} samples")
df.head()

## 2. Data Overview

In [ ]:
# Basic statistics
print("Dataset Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nClass Distribution:")
print(df['label'].value_counts())

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
df['label'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%', labels=['Legitimate', 'Scam'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('')

# Category distribution for scams
scam_df = df[df['label'] == 1]
scam_df['category'].value_counts().plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Scam Categories Distribution')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 3. Text Characteristics Analysis

In [ ]:
# Add text length features
df['message_length'] = df['message'].str.len()
df['word_count'] = df['message'].str.split().str.len()
df['avg_word_length'] = df['message'].apply(lambda x: np.mean([len(word) for word in x.split()]))

# Compare scam vs legitimate
print("Text Characteristics by Class:\n")
print(df.groupby('label')[['message_length', 'word_count', 'avg_word_length']].mean())

In [ ]:
# Visualize text length distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Message length
df[df['label']==0]['message_length'].hist(ax=axes[0,0], bins=30, alpha=0.7, label='Legitimate', color='green')
df[df['label']==1]['message_length'].hist(ax=axes[0,0], bins=30, alpha=0.7, label='Scam', color='red')
axes[0,0].set_title('Message Length Distribution')
axes[0,0].set_xlabel('Length (characters)')
axes[0,0].legend()

# Word count
df[df['label']==0]['word_count'].hist(ax=axes[0,1], bins=30, alpha=0.7, label='Legitimate', color='green')
df[df['label']==1]['word_count'].hist(ax=axes[0,1], bins=30, alpha=0.7, label='Scam', color='red')
axes[0,1].set_title('Word Count Distribution')
axes[0,1].set_xlabel('Words')
axes[0,1].legend()

# Average word length
df[df['label']==0]['avg_word_length'].hist(ax=axes[1,0], bins=30, alpha=0.7, label='Legitimate', color='green')
df[df['label']==1]['avg_word_length'].hist(ax=axes[1,0], bins=30, alpha=0.7, label='Scam', color='red')
axes[1,0].set_title('Average Word Length Distribution')
axes[1,0].set_xlabel('Average Length')
axes[1,0].legend()

# Box plot comparison
df.boxplot(column='message_length', by='label', ax=axes[1,1])
axes[1,1].set_title('Message Length by Class')
axes[1,1].set_xlabel('Class (0=Legitimate, 1=Scam)')
axes[1,1].set_ylabel('Message Length')

plt.tight_layout()
plt.show()

## 4. Pattern Analysis

In [ ]:
# Initialize pattern detector
detector = AustralianPatternDetector()

# Detect patterns in sample scam messages
print("Sample Scam Pattern Detection:\n")
scam_samples = df[df['label']==1].sample(5)

for idx, row in scam_samples.iterrows():
    patterns = detector.detect_all_patterns(row['message'])
    print(f"Message: {row['message'][:80]}...")
    print(f"Category: {row['category']}")
    if patterns:
        for p in patterns:
            print(f"  - {p.pattern_name}: {p.confidence:.2f} confidence")
    else:
        print("  - No specific patterns detected")
    print()

## 5. Australian-Specific Analysis

In [ ]:
# Identify Australian-specific scams
australian_categories = ['ato_impersonation', 'mygov_impersonation', 'banking_scam', 'delivery_scam']
australian_scams = df[df['category'].isin(australian_categories)]

print(f"Australian-specific scams: {len(australian_scams)} ({len(australian_scams)/len(df)*100:.1f}%)")
print("\nBreakdown:")
print(australian_scams['category'].value_counts())

In [ ]:
# Visualize Australian scam categories
plt.figure(figsize=(10, 6))
australian_scams['category'].value_counts().plot(kind='bar', color='skyblue')
plt.title('Australian-Specific Scam Categories', fontsize=14, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Feature Extraction Preview

In [ ]:
# Extract features for a few samples
feature_extractor = FeatureExtractor()

sample = df.sample(1).iloc[0]
features = feature_extractor.extract_all_features(sample['message'])

print("Sample Message:")
print(sample['message'])
print(f"\nLabel: {'Scam' if sample['label']==1 else 'Legitimate'}")
print("\nExtracted Features:")
for key, value in features.items():
    print(f"  {key}: {value}")

## 7. Key Insights

### Summary
- Dataset contains a balanced mix of scam and legitimate messages
- Scam messages tend to be [shorter/longer] than legitimate ones
- Australian-specific patterns account for X% of scams
- Most common scam category: [category]

### Next Steps
1. Proceed to preprocessing and feature engineering
2. Train ML models (XGBoost and BERT)
3. Evaluate performance on test set
4. Deploy best performing model